# 4. Robustness across sensors, noise, and observability

**Question:** Does the local–scrambled comparison persist when observations become noisy, ambiguous, or image-like, and does it persist under a non-collapse-oriented objective?

This workflow combines numerical-sensor noise/aliasing checks with a 16 × 16 image–sensor comparison. Keep the clean setting as an anchor, but do not count a reused identical clean run as independent replication.

An aliased sensor maps antipodal states to exactly the same observation distribution. A memoryless encoder cannot recover that missing identity. Report both the original coordinate probe and the accessible doubled-angle quotient probe. Images here are generated Gaussian markers, not real-world validation.

Results stay in memory by default. The opt-in storage cell is the only route to result files; notebook-editor output saving is a separate setting.

In [ ]:
from pathlib import Path
import sys
import subprocess
from zipfile import ZipFile

REPO_URL = 'https://github.com/AlexM3020/The-Role-of-Dynamical-Structure-in-Cross-Modal-Representation-Learning.git'
REPO_REF = 'main'  # For a frozen run, choose a commit before starting a fresh runtime.
SOURCE_ARCHIVE = None  # Optional uploaded source ZIP; not needed in Colab with GitHub access.
INSTALL_DEPENDENCIES = False  # Enable only if the runtime lacks compatible dependencies.
cloud = next((p for p in (Path('/content'), Path('/kaggle/working')) if p.exists()), None)
if SOURCE_ARCHIVE is not None:
    if cloud is None:
        raise RuntimeError('Use the local checkout here; source extraction is for cloud runtimes.')
    destination = cloud / 'dynalign_uploaded_source'
    with ZipFile(SOURCE_ARCHIVE) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive member')
        archive.extractall(destination)
    candidates = [destination]
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'src/dynalign/revised.py').exists()), None)
if ROOT is None and SOURCE_ARCHIVE is None and cloud is not None:
    destination = cloud / 'dynalign_github'
    if not destination.exists():
        subprocess.check_call(['git', 'clone', '--quiet', REPO_URL, str(destination)])
        subprocess.check_call(['git', '-C', str(destination), 'fetch', '--quiet', 'origin', REPO_REF])
        subprocess.check_call(['git', '-C', str(destination), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'])
    ROOT = destination
if ROOT is None or not (ROOT / 'src/dynalign/revised.py').exists():
    raise FileNotFoundError('Open the current checkout locally, or use a fresh Colab/Kaggle runtime with Internet enabled.')
# Existing checkouts are deliberately reused, not reset or silently updated.
# Restart in a fresh runtime to switch REPO_REF or load a newer GitHub revision.
if 'dynalign' in sys.modules and Path(sys.modules['dynalign'].__file__).resolve().parents[2] != ROOT.resolve():
    raise RuntimeError('Another source version is already imported; restart the runtime before switching.')
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
revision = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True)
print('Source revision:', revision.stdout.strip() if revision.returncode == 0 else 'uploaded source bundle')
print('Source setup only: result saving remains disabled by default.')

from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dynalign.revised import FitSettings, EvalSettings, SyntheticSpec, run_synthetic, preview_plan, paired_contrasts, bundle_zip
from dynalign.worlds import image_table

## Experiments to try

1. **Clean numerical anchor:** noise 0.05 in both sensors, at 2,048 transitions.
2. **Noisy measurements:** increase numerical noise to 0.30 while keeping it identical across local/scrambled conditions.
3. **Partial observability:** alias antipodal states in sensor B; compare original-coordinate and doubled-angle quotient recovery.
4. **Different observation type:** a tiny CNN on generated images versus the numerical MLP; image noise 0.03, numerical noise 0.05.
5. **Objective cross-check:** run baseline and the candidate frozen after workflow 2 for each setting. Start with 10 worlds and two repeats for the expanded grid.

Exact information matching applies within each observation setting, not between different noise levels or sensor types. The pixel case tests synthetic modality robustness only. Two clean runs with identical settings/seeds are reused evidence, not independent replications. Avoid interpreting fixed circular-coordinate probes alone, since that target favors the local topology.

In [ ]:
RUN_TRAINING = False
PHASE = 'development'
WORLD_IDS, REPEAT_IDS = [300], [0]  # Expanded: 10 fresh worlds and two repeats.
MASTER_SEED = 20261108
CONDITIONS = ['local', 'scrambled']
N_PAIRS = 256  # Expanded robustness anchor: 2048
N_STATES, SENSOR_WIDTH, EMBEDDING_DIM = 32, 12, 4
WIDTH, ARCHITECTURE = 64, 'mlp'
IMAGE_SIDE, IMAGE_NOISE = 16, 0.03
DEVICE, CPU_THREADS = 'auto', 2
SAVE_RESULTS, OUTPUT_DIR = False, None
FIT = FitSettings(steps=80, checkpoints=(40, 80), batch_size=128, lr=1e-3,
                  weight_decay=1e-4, invariance_weight=25, variance_weight=25,
                  covariance_weight=1, identity_weight=25, variance_target=1,
                  epsilon=1e-4, clip_grad=5, log_every=40, checkpoint_every=100)
EVAL = EvalSettings(calibration_repeats=8, test_repeats=16, ridge=1e-3,
                    collapse_variance=1e-6, collapse_rank_fraction=0.5, batch_size=512)
OBSERVATIONS = {
    'clean': dict(noise=0.05, aliased_b=False, pixels=False),
    'noisy': dict(noise=0.30, aliased_b=False, pixels=False),
    'aliased': dict(noise=0.05, aliased_b=True, pixels=False),
    'pixels': dict(noise=0.05, aliased_b=False, pixels=True),
}
ACTIVE_OBSERVATIONS = ['clean', 'noisy', 'aliased', 'pixels']
REGIMES = {
    'baseline': dict(objective='vicreg', invariance_weight=25, variance_weight=25, covariance_weight=1),
    'identity_candidate': dict(objective='cov_identity', invariance_weight=25, identity_weight=25),
}
ACTIVE_REGIMES = ['baseline', 'identity_candidate']
if SAVE_RESULTS and OUTPUT_DIR is None:
    raise ValueError('Saving requires an explicit destination.')
specs = [SyntheticSpec(experiment='robustness_' + observation, phase=PHASE, world=w, repeat=r,
                      master_seed=MASTER_SEED, condition=c, n_states=N_STATES, n_pairs=N_PAIRS,
                      sensor_width=SENSOR_WIDTH, architecture=ARCHITECTURE, width=WIDTH,
                      dim=EMBEDDING_DIM, image_side=IMAGE_SIDE, image_noise=IMAGE_NOISE,
                      regime=regime, fit=replace(FIT, **REGIMES[regime]), evaluation=EVAL,
                      **OBSERVATIONS[observation])
         for w in WORLD_IDS for r in REPEAT_IDS for observation in ACTIVE_OBSERVATIONS
         for regime in ACTIVE_REGIMES for c in CONDITIONS]
display(preview_plan(specs))
print(f'{2 * len(specs)} encoder fits. Training enabled: {RUN_TRAINING}')

## Independent learning for each observation setting

Each comparison changes the transition rule while holding its sensor setting fixed. Encoders train on their own observation pairs only. The image encoder is fixed across the pixel comparison, as is the vector encoder; parameter counts are recorded, not assumed equal across modalities.

**Edit loss weights in `REGIMES`, which override the matching `FIT` defaults; edit sensor noise/aliasing in `OBSERVATIONS`.** The displayed plan lists the merged settings. No hidden profile substitutes different values.

The run returns original-coordinate, quotient, and transition-specific spectral probes. A full-state spectral target can be inaccessible under aliasing; interpret it alongside the privileged sensor reference from workflow 1. Strong covariance regularization cannot create state information that the input lacks.

In [ ]:
bundle = None
if RUN_TRAINING:
    bundle = run_synthetic(specs, device=DEVICE, threads=CPU_THREADS,
                           output_dir=OUTPUT_DIR if SAVE_RESULTS else None)
    results = bundle.table
    display(results[['experiment', 'regime', 'condition', 'world', 'step', 'cka',
                     'coordinate_r2_b', 'quotient_r2_b', 'any_collapse',
                     'estimated_signal_fraction_a', 'estimated_signal_fraction_b']])
else:
    print('Training disabled. Edit the configuration and enable RUN_TRAINING when ready.')

## Assess robustness without conflating different questions

Compute local-minus-scrambled contrasts separately for each observation setting and objective. Do not compare aliased/full-state scores as if all factors were observable, or treat a clean baseline reused elsewhere as independent evidence.

A preserved gap after the rank diagnostics is stronger than repeating the original low-rank pattern. A changed or reversed effect is also a result worth retaining. This workflow retrains for each setting; it is not a test-time-corruption or unseen-dynamics experiment.

Use world-level uncertainty and show the full distribution, rather than treating image frames or initialization repeats as independent worlds.

In [ ]:
if bundle is not None:
    contrasts = paired_contrasts(results, metric='cka', group_by=['experiment', 'regime', 'n_pairs', 'step'])
    display(contrasts)
    last = results[results.step == FIT.steps]
    display(last[['experiment', 'regime', 'condition', 'world', 'coordinate_r2_a', 'coordinate_r2_b',
                  'quotient_r2_a', 'quotient_r2_b', 'estimated_signal_fraction_a',
                  'estimated_signal_fraction_b', 'effective_rank_a', 'effective_rank_b', 'any_collapse']])
    summary = last.groupby(['experiment', 'regime', 'condition']).cka.mean().unstack('condition')
    summary.plot.bar(figsize=(11, 4), ylabel='Mean CKA', title='Observation-specific comparisons (descriptive means)')
    plt.tight_layout()
    plt.show()
else:
    images = image_table(N_STATES, IMAGE_SIDE)
    fig, axes = plt.subplots(1, 4, figsize=(8, 2))
    for ax, image in zip(axes, images[::max(1, N_STATES // 4)]):
        ax.imshow(image[0], cmap='gray'); ax.axis('off')
    plt.show()
    print('Observation preview only; no models trained or results saved.')

## Optional export

Results and diagnostics stay in RAM. ZIP preparation below is also memory-only. Saving/checkpointing happens only when `SAVE_RESULTS=True` and `OUTPUT_DIR` is explicitly supplied. No default local directory, `savefig`, or automatic browser download is used.

For cloud sessions, deliberately choose durable storage before a long run or manually export at the end. A displayed download link does not download until clicked. Editor autosave of cell outputs is a separate setting; clear outputs before saving the notebook if required.

In [ ]:
SHOW_DOWNLOAD_LINK = False
EXPORT_BYTES = bundle_zip(bundle) if bundle is not None else None
if EXPORT_BYTES is not None:
    print(f'{len(EXPORT_BYTES):,} export bytes in RAM. Saving enabled: {SAVE_RESULTS}')
    if SHOW_DOWNLOAD_LINK:
        import base64
        from IPython.display import HTML
        display(HTML('<a download="observation_robustness.zip" href="data:application/zip;base64,' +
                     base64.b64encode(EXPORT_BYTES).decode() + '">Click to download</a>'))
else:
    print('No results to export; nothing saved or downloaded.')